# 03 - Exploratory Data Analysis
Twenty analytical questions on the **cleaned** dataset. Every chart has a title, axis labels and a
short interpretation. Interpretations describe **associations** only - correlation is not causation.

*Track-level questions use one row per `track_id`; genre-level questions use every (track, genre) row.
Rankings only include artists/genres above the minimum track counts set in `config/config.yaml`.*

In [ ]:
import sys, warnings
from pathlib import Path

# Make `src` importable whether Jupyter starts in the project root or in notebooks/
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import load_config, resolve_path

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.3f}".format)

cfg = load_config()

import seaborn as sns
from src import eda, insights
from src.data_loader import load_processed

df = load_processed(cfg)
FIG_DIR = resolve_path(cfg["paths"]["reports_dir"]) / "figures"
MIN_ARTIST = cfg["analysis"]["min_tracks_per_artist"]
MIN_GENRE = cfg["analysis"]["min_tracks_per_genre"]
SEED = cfg["analysis"]["random_seed"]
SAMPLE_N = cfg["analysis"]["plot_sample_size"]

def show(fig, name):
    eda.save_figure(fig, name, FIG_DIR)
    plt.show()

print(f"Loaded {len(df):,} rows / {df['track_id'].nunique():,} unique tracks")

## Q1. What are the top 10 most popular tracks?
*Interpretation lines are printed from the computed values, so they always match your data.*

In [ ]:
top = eda.top_tracks(df, 10)
top["label"] = top["track_name"] + " - " + top["primary_artist"]
display(top[["track_name", "primary_artist", "track_genre", "popularity"]])
show(eda.bar_chart(top, "label", "popularity", "Top 10 Most Popular Tracks", "Popularity score (0-100)", "Track - artist"), "q01_top_tracks")
print(f"Interpretation: the highest popularity score is {top['popularity'].max()}; "
      f"{int((top['popularity'] == top['popularity'].max()).sum())} of the top 10 share that score.")

## Q2. Who are the top 10 artists by average popularity?
*Interpretation lines are printed from the computed values, so they always match your data.*

In [ ]:
artists = eda.top_artists_by_avg_popularity(df, 10, MIN_ARTIST)
display(artists)
show(eda.bar_chart(artists, "primary_artist", "avg_popularity", f"Top 10 Artists by Average Popularity (min. {MIN_ARTIST} tracks)", "Average popularity", "Artist"), "q02_top_artists")
if len(artists):
    print(f"Interpretation: {artists.iloc[0]['primary_artist']} leads with an average of {artists.iloc[0]['avg_popularity']} "
          f"across {int(artists.iloc[0]['track_count'])} tracks. Artists with fewer than {MIN_ARTIST} tracks are excluded.")

## Q3. Which genres have the highest average popularity?
*Interpretation lines are printed from the computed values, so they always match your data.*

In [ ]:
g = eda.genre_metric_ranking(df, "popularity", 10, MIN_GENRE)
display(g)
show(eda.bar_chart(g, "track_genre", "avg_popularity", "Top 10 Genres by Average Popularity", "Average popularity", "Genre"), "q03_genre_popularity")
print(f"Interpretation: '{g.iloc[0]['track_genre']}' ranks first ({g.iloc[0]['avg_popularity']}). Genre averages depend on which tracks the dataset sampled.")

## Q4. Which genres have the highest danceability?
*Interpretation lines are printed from the computed values, so they always match your data.*

In [ ]:
g = eda.genre_metric_ranking(df, "danceability", 10, MIN_GENRE)
display(g)
show(eda.bar_chart(g, "track_genre", "avg_danceability", "Top 10 Genres by Average Danceability", "Average danceability (0-1)", "Genre"), "q04_genre_danceability")
print(f"Interpretation: '{g.iloc[0]['track_genre']}' is the most danceable genre on average ({g.iloc[0]['avg_danceability']}).")

## Q5. Which genres have the highest energy?
*Interpretation lines are printed from the computed values, so they always match your data.*

In [ ]:
g = eda.genre_metric_ranking(df, "energy", 10, MIN_GENRE)
display(g)
show(eda.bar_chart(g, "track_genre", "avg_energy", "Top 10 Genres by Average Energy", "Average energy (0-1)", "Genre"), "q05_genre_energy")
print(f"Interpretation: '{g.iloc[0]['track_genre']}' has the highest average energy ({g.iloc[0]['avg_energy']}).")

## Q6. What is the average song duration?
*Interpretation lines are printed from the computed values, so they always match your data.*

In [ ]:
d = eda.duration_summary(df)
display(pd.Series(d, name="minutes").to_frame())
print(f"Interpretation: the mean duration is {d['mean_min']} min and the median {d['median_min']} min; "
      f"{'mean > median, so a few long tracks pull the average up' if d['mean_min'] > d['median_min'] else 'mean is not above the median'}.")

## Q7. Which songs have the highest energy?
*Interpretation lines are printed from the computed values, so they always match your data.*

In [ ]:
t = eda.top_tracks_by_feature(df, "energy", 10)
t["label"] = t["track_name"] + " - " + t["primary_artist"]
display(t.drop(columns="label"))
show(eda.bar_chart(t, "label", "energy", "Tracks with the Highest Energy", "Energy (0-1)", "Track - artist"), "q07_top_energy")
print(f"Interpretation: the top energy value is {t['energy'].max()}; ties at the maximum are common for a bounded 0-1 feature.")

## Q8. Which songs have the highest danceability?
*Interpretation lines are printed from the computed values, so they always match your data.*

In [ ]:
t = eda.top_tracks_by_feature(df, "danceability", 10)
t["label"] = t["track_name"] + " - " + t["primary_artist"]
display(t.drop(columns="label"))
show(eda.bar_chart(t, "label", "danceability", "Tracks with the Highest Danceability", "Danceability (0-1)", "Track - artist"), "q08_top_danceability")
print(f"Interpretation: the top danceability value is {t['danceability'].max()}.")

## Q9-Q11. Relationship between popularity and energy / danceability / acousticness
*Hexbin plots show where tracks concentrate; Pearson r measures **linear** association only.*

In [ ]:
corr = eda.popularity_correlations(df, ["energy", "danceability", "acousticness"], seed=SEED)
display(corr)
tracks = eda.unique_tracks(df)
for n, feature in [(9, "energy"), (10, "danceability"), (11, "acousticness")]:
    fig = eda.density_scatter(tracks, feature, "popularity",
                              f"Popularity vs {feature.capitalize()} (density of tracks)",
                              f"{feature.capitalize()} (0-1)", "Popularity score (0-100)", SAMPLE_N, SEED)
    show(fig, f"q{n:02d}_popularity_vs_{feature}")
    row = corr[corr["feature"] == feature].iloc[0]
    print(f"Q{n} interpretation: {feature} and popularity show a {insights.describe_correlation(row['pearson_r'])} "
          f"linear association (Pearson r = {row['pearson_r']}, Spearman rho = {row['spearman_rho']}). "
          "This does not show that the feature causes popularity.")

## Q12. Which artists have the most tracks?
*Interpretation lines are printed from the computed values, so they always match your data.*

In [ ]:
a = eda.artist_track_counts(df, 10)
display(a)
show(eda.bar_chart(a, "primary_artist", "track_count", "Artists with the Most Tracks", "Number of distinct tracks", "Artist"), "q12_artist_track_counts")
print(f"Interpretation: {a.iloc[0]['primary_artist']} has the most tracks ({int(a.iloc[0]['track_count'])}).")

## Q13. What percentage of tracks are explicit?
*Interpretation lines are printed from the computed values, so they always match your data.*

In [ ]:
share = eda.explicit_share(df)
if not share["available"]:
    print("This dataset has no 'explicit' column - Q13 and Q14 are skipped.")
else:
    pct = share["explicit_pct"]
    data = pd.DataFrame({"type": ["Explicit", "Not explicit"], "pct": [pct, 100 - pct]})
    show(eda.bar_chart(data, "type", "pct", "Share of Explicit vs Non-Explicit Tracks", "Share of tracks (%)", "Track type"), "q13_explicit_share")
    print(f"Interpretation: {pct}% of tracks ({share['explicit_tracks']:,} of {share['tracks_with_flag']:,}) are marked explicit.")

## Q14. How does explicit content relate to popularity?
*Interpretation lines are printed from the computed values, so they always match your data.*

In [ ]:
if not eda.explicit_share(df)["available"]:
    print("Skipped: no 'explicit' column in this dataset.")
else:
    summary = eda.explicit_vs_popularity(df)
    display(summary)
    plot_df = eda.unique_tracks(df).dropna(subset=["explicit"]).copy()
    plot_df["explicit"] = plot_df["explicit"].map({0: "Not explicit", 1: "Explicit"})
    show(eda.box_by_group(plot_df, "explicit", "popularity", "Popularity by Explicit Flag", "Track type", "Popularity score (0-100)"), "q14_explicit_popularity")
    means = summary.set_index("explicit")["mean_popularity"]
    if {0, 1} <= set(means.index):
        print(f"Interpretation: mean popularity is {means[1]} for explicit vs {means[0]} for non-explicit tracks "
              f"(difference {means[1] - means[0]:+.2f}). This is a descriptive difference, not evidence that explicitness changes popularity.")

## Q15. What is the distribution of song popularity?
*Interpretation lines are printed from the computed values, so they always match your data.*

In [ ]:
pop = eda.unique_tracks(df)["popularity"]
show(eda.histogram(pop, "Distribution of Track Popularity", "Popularity score (0-100)", "Number of tracks", bins=50), "q15_popularity_distribution")
print(f"Interpretation: mean {pop.mean():.2f}, median {pop.median():.2f}, std {pop.std():.2f}; "
      f"{(pop == 0).mean() * 100:.1f}% of tracks have a popularity of exactly 0.")

## Q16. What is the distribution of song duration?
*Interpretation lines are printed from the computed values, so they always match your data.*

In [ ]:
dur = eda.unique_tracks(df)["duration_min"]
upper = dur.quantile(0.99)
show(eda.histogram(dur[dur <= upper], "Distribution of Track Duration (up to the 99th percentile)", "Duration (minutes)", "Number of tracks", bins=60), "q16_duration_distribution")
print(f"Interpretation: median {dur.median():.2f} min; 99% of tracks are shorter than {upper:.2f} min. "
      f"The longest track is {dur.max():.1f} min (extreme values are excluded from the plot only).")

## Q17. Which genres have the highest instrumentalness?
*Interpretation lines are printed from the computed values, so they always match your data.*

In [ ]:
g = eda.genre_metric_ranking(df, "instrumentalness", 10, MIN_GENRE)
display(g)
show(eda.bar_chart(g, "track_genre", "avg_instrumentalness", "Top 10 Genres by Average Instrumentalness", "Average instrumentalness (0-1)", "Genre"), "q17_genre_instrumentalness")
print(f"Interpretation: '{g.iloc[0]['track_genre']}' is the most instrumental genre on average ({g.iloc[0]['avg_instrumentalness']}).")

## Q18. What is the correlation between audio features?
*Interpretation lines are printed from the computed values, so they always match your data.*

In [ ]:
cm = eda.feature_correlation_matrix(df)
show(eda.heatmap(cm, "Correlation Matrix of Audio Features and Popularity", "Feature", "Feature"), "q18_correlation_heatmap")
pairs = cm.where(np.triu(np.ones(cm.shape, dtype=bool), k=1)).stack().sort_values(key=abs, ascending=False)
print("Strongest feature pairs (by |r|):")
for (a, b), r in pairs.head(3).items():
    print(f"  {a} vs {b}: r = {r:.3f} ({insights.describe_correlation(r)})")
print("Interpretation: features can be correlated with each other (multicollinearity), so a single feature's link with popularity should not be read in isolation.")

## Q19. Which genres have the highest average valence?
*Interpretation lines are printed from the computed values, so they always match your data.*

In [ ]:
g = eda.genre_metric_ranking(df, "valence", 10, MIN_GENRE)
display(g)
show(eda.bar_chart(g, "track_genre", "avg_valence", "Top 10 Genres by Average Valence (Positivity)", "Average valence (0-1)", "Genre"), "q19_genre_valence")
print(f"Interpretation: '{g.iloc[0]['track_genre']}' has the highest average valence ({g.iloc[0]['avg_valence']}).")

## Q20. What are the key characteristics of popular songs?
*Interpretation lines are printed from the computed values, so they always match your data.*

In [ ]:
profile = eda.popular_vs_rest_profile(df, cfg["analysis"]["popular_quantile"])
meta = profile.attrs
print(f"'Popular' = popularity >= {meta['threshold']:.0f} ({meta['popular_count']:,} tracks); compared with {meta['rest_count']:,} other tracks.")
display(profile)
plot = profile.sort_values("std_difference")
show(eda.bar_chart(plot, "feature", "std_difference", "Popular vs Other Tracks: Standardised Feature Difference", "Difference in standard deviations (popular minus other)", "Audio feature"), "q20_popular_profile")
top_feature = profile.reindex(profile["std_difference"].abs().sort_values(ascending=False).index).iloc[0]
print(f"Interpretation: the largest standardised gap is in {top_feature['feature']} "
      f"({top_feature['popular_mean']:.3f} vs {top_feature['rest_mean']:.3f}). These are associations in this dataset, not causes.")

## Summary of findings (generated from the computed facts)

In [ ]:
facts = insights.compute_facts(df, MIN_ARTIST, MIN_GENRE, cfg["analysis"]["popular_quantile"])
for i, text in enumerate(insights.generate_insights(facts, 10), 1):
    print(f"{i}. {text}\n")

*All figures were saved to `reports/figures/`.*